In [69]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
import selenium.webdriver
from bs4 import BeautifulSoup
import re
import time
import csv
from urllib.parse import urljoin, urlparse


In [70]:
chrome_options = Options()
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option('useAutomationExtension', False)

In [71]:
chrome_driver_path = r'chromedriver.exe'
service = Service(chrome_driver_path)

In [72]:
def discover_car_listing_url(base_url, driver_path):
    driver = webdriver.Chrome(service=service)
    driver.get(base_url)
    print(f"در حال بررسی صفحه اصلی: {base_url}")
    
    wait = WebDriverWait(driver, 15)
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    driver.quit()

    keywords = ['car', 'cars', 'auto', 'automobile', 'buy-car', 'sell-car', 'used-cars', 'new-cars', 'خودرو', 'ماشین']
    found_links = []

    for link in soup.find_all('a', href=True):
        href = link['href']
        full_url = urljoin(base_url, href)
        link_text = link.get_text(strip=True).lower()
        href_lower = href.lower()

        score = 0
        
        for keyword in keywords:
            if keyword in href_lower:
                score += 3  
            if keyword in link_text:
                score += 2  

        if re.search(r'/(car|auto|vehicle)s?/?$', href_lower): score += 5
        if re.search(r'/(buy|sell|search|list|find)-?(car|auto)', href_lower): score += 4
        
        if score > 0:
            found_links.append((score, full_url, link_text))
            print(f"  لینک احتمالی پیدا شد (امتیاز {score}): {full_url} - متن: {link_text}")

    if found_links:
        found_links.sort(key=lambda x: x[0], reverse=True)
        best_url = found_links[0][1]
        print(f"\n✅ بهترین آدرس پیدا شده: {best_url}")
        return best_url
    else:
        print("\n❌ هیچ لینک مرتبطی با خودرو پیدا نشد!")
        return None

In [73]:
def save_car_to_csv(data, filename="bama_car_1385plus.csv"):
    fieldnames = ["title", "year", "price", "url", "raw_detail"]
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in data:
            writer.writerow(row)
    print(f"✅ {len(data)} ردیف در {filename} ذخیره شد.")

In [74]:
def normalize(text):
    if not text:
        return ""
    return text.replace("\u200c", " ").translate(fa_to_en).strip()

def extract_year(text):
    text = normalize(text)
    m = re.search(r'\b(13[8-9]\d|140\d)\b', text)
    return int(m.group(0)) if m else None

def extract_price(text):
    text = normalize(text)
    if "قیمت توافقی" in text:
        return "قیمت توافقی"
    m = re.search(r'([\d,]+)\s*تومان', text)
    if m:
        return f'{m.group(1)} تومان'
    return ""

def extract_km(text):
    text = normalize(text)
    if "صفر کیلومتر" in text:
        return "صفر کیلومتر"
    m = re.search(r'([\d,]+)\s*کیلومتر', text)
    if m:
        return f'{m.group(1)} کیلومتر'
    return ""

def extract_city(text):
    text = normalize(text)
    m = re.search(r'([آ-یA-Za-z\s]+?)\s*،', text)
    if m:
        city = m.group(1).strip()
        if len(city) <= 25:
            return city
    return ""


In [76]:
if __name__ == "__main__":
    base_site = "https://bama.ir"
    chromedriver_path = r'C:\chromedriver\chromedriver.exe'
    
    car_listing_url = discover_car_listing_url(base_site, chrome_driver_path)
    
    if car_listing_url:
        print(f"\n🔗 برای اسکرپینگ ماشین‌ها، از آدرس زیر استفاده کن: {car_listing_url}")
    else:
        print("برنامه نتوانست آدرس بخش ماشین‌ها را پیدا کند.")

در حال بررسی صفحه اصلی: https://bama.ir
  لینک احتمالی پیدا شد (امتیاز 10): https://bama.ir/car - متن: خرید خودرو
  لینک احتمالی پیدا شد (امتیاز 10): https://bama.ir/sell/car - متن: ثبت آگهی خودرو
  لینک احتمالی پیدا شد (امتیاز 2): https://bama.ir/price - متن: قیمت روز خودرو
  لینک احتمالی پیدا شد (امتیاز 2): https://bama.ir/calculator - متن: تخمین قیمت خودرو
  لینک احتمالی پیدا شد (امتیاز 5): https://bama.ir/car-reviews - متن: مقایسه و مشخصات فنی خودرو
  لینک احتمالی پیدا شد (امتیاز 2): https://bama.ir/news - متن: اخبار خودرو
  لینک احتمالی پیدا شد (امتیاز 10): https://bama.ir/alert/car - متن: گوش‌به‌زنگ خودرو
  لینک احتمالی پیدا شد (امتیاز 2): https://bama.ir/dealer/sign-up - متن: ثبت نام نمایشگاه خودرو
  لینک احتمالی پیدا شد (امتیاز 10): https://bama.ir/dealer/car - متن: نمایشگاه‌های خودرو
  لینک احتمالی پیدا شد (امتیاز 3): https://bama.ir/price-alert?type=car - متن: سرویس پیامکی قیمت
  لینک احتمالی پیدا شد (امتیاز 2): https://bama.ir/news/sale-offers - متن: شرایط فروش خودروسازها
  

In [85]:
driver = webdriver.Chrome(service=service, options=chrome_options)
car_name=input('لطفا نام مماشین مورد نظر را وارد کنید ')
driver.maximize_window()
car_listing_url2 =car_listing_url+ '/'+car_name
driver.get(car_listing_url2)
print("\nصفحه لیست خودروها باز شد.")

soup = BeautifulSoup(full_html, "html.parser")

fa_to_en = str.maketrans("۰۱۲۳۴۵۶۷۸۹", "0123456789")


cars = []

cards = soup.select('article a[href^="/car/detail-"]')

for a in cards:
    try:
        card_text = a.get_text(" ", strip=True)
        card_text = normalize(card_text)

        href = a.get("href", "")
        url = "https://bama.ir" + href if href.startswith("/") else href

        title = ""
        title_tag = a.find(["h2", "h3", "h4"])
        if title_tag:
            title = normalize(title_tag.get_text(" ", strip=True))
        else:
            parts = [p.strip() for p in re.split(r'\s{2,}| \| | - ', card_text) if p.strip()]
            title = parts[0] if parts else card_text[:100]

        year = extract_year(card_text)
        if year is None or year < 1385:
            continue

        price = extract_price(card_text)
        km = extract_km(card_text)
        city = extract_city(card_text)

        model = ""
        if "پلاس" in card_text:
            m = re.search(r'(پلاس\s+[^\d،]+)', card_text)
            if m:
                model = m.group(1).strip()
        elif "EF7" in card_text or "XU7P" in card_text or "دنده ای" in card_text:
            m = re.search(r'(EF7|XU7P|دنده ای|اتوماتیک)', card_text)
            if m:
                model = m.group(1).strip()

        cars.append({
            "title": title,
            "model": model,
            "year": year,
            "km": km,
            "city": city,
            "price": price,
            "url": url,
            "raw_text": card_text
        })

        if len(cars) >= 50:
            break

    except Exception as e:
        print("خطا در یک کارت:", e)

print("تعداد نهایی:", len(cars))

csv_file = car_name+"_1385_plus.csv"
with open(csv_file, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["title", "model", "year", "km", "city", "price", "url", "raw_text"]
    )
    writer.writeheader()
    writer.writerows(cars)

print("فایل ذخیره شد:", csv_file)
save_car_to_csv(cars_data)

لطفا نام مماشین مورد نظر را وارد کنید  samand



صفحه لیست خودروها باز شد.
تعداد نهایی: 50
فایل ذخیره شد: samand_1385_plus.csv
✅ 0 ردیف در bama_car_1385plus.csv ذخیره شد.


In [87]:
df= ps.read_csv(csv_file)
df

,title,model,year,km,city,price,url,raw_text
0,"4 سمند، سورن پلاس EF7 بنزینی 1401 . کارکرد 56,...",پلاس EF,1401,"56,500 کیلومتر",سمند,"1,680,000,000 تومان",https://bama.ir/car/detail-lh05iidr-samand-sor...,"4 سمند، سورن پلاس EF7 بنزینی 1401 . کارکرد 56,..."
1,1 سمند، سورن پلاس EF7 بنزینی 1403 . صفر کیلومت...,پلاس EF,1403,صفر کیلومتر,سمند,"550,000,000 تومان",https://bama.ir/car/detail-nlu7hatn-samand-sor...,1 سمند، سورن پلاس EF7 بنزینی 1403 . صفر کیلومت...
2,1 سمند، سورن پلاس XU7P بنزینی 1404 . صفر کیلوم...,پلاس XU,1404,صفر کیلومتر,سمند,"1,850,000,000 تومان",https://bama.ir/car/detail-ia9xfjbg-samand-sor...,1 سمند، سورن پلاس XU7P بنزینی 1404 . صفر کیلوم...
3,1 سمند، سورن پلاس XU7P بنزینی 1405 . صفر کیلوم...,پلاس XU,1405,صفر کیلومتر,سمند,"1,940,000,000 تومان",https://bama.ir/car/detail-oulrb1ts-samand-sor...,1 سمند، سورن پلاس XU7P بنزینی 1405 . صفر کیلوم...
4,"سمند، X7 دنده ای 1387 . کارکرد 85,000 کیلومتر ...",دنده ای,1387,"85,000 کیلومتر",سمند,قیمت توافقی,https://bama.ir/car/detail-knvmctub-samand-x7-...,"سمند، X7 دنده ای 1387 . کارکرد 85,000 کیلومتر ..."
5,4 سمند، سورن پلاس EF7 دوگانه سوز 1404 . صفر کی...,پلاس EF,1404,صفر کیلومتر,سمند,"700,000,000 تومان",https://bama.ir/car/detail-m9ozi5gq-samand-sor...,4 سمند، سورن پلاس EF7 دوگانه سوز 1404 . صفر کی...
6,سمند، سورن پلاس XU7P بنزینی 1405 . صفر کیلومتر...,پلاس XU,1405,صفر کیلومتر,سمند,قیمت توافقی,https://bama.ir/car/detail-qzxxfz6i-samand-sor...,سمند، سورن پلاس XU7P بنزینی 1405 . صفر کیلومتر...
7,سمند، LX EF7 دوگانه سوز 1401 . صفر کیلومتر 3 س...,EF7,1401,صفر کیلومتر,سمند,قیمت توافقی,https://bama.ir/car/detail-zvgid69u-samand-lx-...,سمند، LX EF7 دوگانه سوز 1401 . صفر کیلومتر 3 س...
8,3 سمند، سورن پلاس XU7P بنزینی 1405 . صفر کیلوم...,پلاس XU,1405,صفر کیلومتر,سمند,قیمت توافقی,https://bama.ir/car/detail-v69tyc5z-samand-sor...,3 سمند، سورن پلاس XU7P بنزینی 1405 . صفر کیلوم...
9,سمند، سورن پلاس XU7P بنزینی 1405 . صفر کیلومتر...,پلاس XU,1405,صفر کیلومتر,سمند,"1,990,000,000 تومان",https://bama.ir/car/detail-zdgzmsau-samand-sor...,سمند، سورن پلاس XU7P بنزینی 1405 . صفر کیلومتر...


<div dir='rtl'>
    در ابتدا به ادرس اصلی سایت می‌رویم و سپس سایت های زیر مجموعه را می‌گردیم تا محتمل‌ترین ادرس که مربوط به خرید و فروش ماشین باشد را پیدا کنیم.
    <br>
    این ادرس جدید براساس امتیازدهی و وجود اسامی مانند car , فروش , خرید و برخی کلمات مشابه در ادرس  url و صفحه می‌باشد.
    <br>
    در مرحله بعد نام ماشین را از کاربر گرفته و به url  اضافه می‌کنیم.
    <br>
    <b>نکته:</b>
    نام ماشین‌ها در صورت داشتن بخش اختصاصی در سایت با نام url جمع می‌شود و دوباره ادرس جدید در chromedriver جستجو می‌شود.
    <br>
    کد HTML در هنگام اسکرول کردن به صورت خودکار در یک استرینگ جمع ذخیره می‌شود و در ادامه با استفاده از parser در ستون های جدا به اندازه 50 ماشین ذخیره می‌شود.
    <br>
    در ادامه داده های ذخیره شده را در یک فایل با اسم زیر ذخیره می‌کنیم:
    <br>
</div>
car_name+"_1385_plus.csv"